# RDKit Descriptor Preprocessing

This notebook preprocesses the generated RDKit descriptor dataset before modelling. Missing descriptor values are imputed using medians fitted on the training set, constant features are removed, and highly correlated descriptors are filtered using the training data only. The same transformations are then applied to the validation and held-out scaffold test sets, producing the final preprocessed dataset and selected descriptor list used by the baseline models.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold

MODEL_FILE = "model_dataset.csv"

model_df = pd.read_csv(MODEL_FILE)

print("Dataset loaded successfully.")
print("Dataset shape:", model_df.shape)

print("\nMolecules in each split:")
print(model_df["split"].value_counts())

In [ ]:
# Identify descriptors and create train, validation and test data These columns contain molecule information
information_columns = [
    "canonical_smiles",
    "smiles",
    "gsk3b_score",
    "jnk3_score",
    "number_of_records",
    "gsk3b_candidate",
    "jnk3_candidate",
    "dual_candidate",
    "docking_category",
    "scaffold",
    "split"
]

# Everything else is an RDKit descriptor
descriptor_columns = [
    column for column in model_df.columns
    if column not in information_columns
]

# Separate the descriptors according to the saved scaffold split
X_train = model_df.loc[
    model_df["split"] == "train",
    descriptor_columns
].copy()

X_validation = model_df.loc[
    model_df["split"] == "validation",
    descriptor_columns
].copy()

X_test = model_df.loc[
    model_df["split"] == "test",
    descriptor_columns
].copy()

print("Number of original descriptors:", len(descriptor_columns))

print("\nTraining shape:", X_train.shape)
print("Validation shape:", X_validation.shape)
print("Test shape:", X_test.shape)

In [ ]:
# Fill missing descriptor values using training medians
# Create the median imputer
imputer = SimpleImputer(strategy="median")

# Learn the median values from the training set only
X_train_imputed_array = imputer.fit_transform(X_train)

# Apply the same training medians to validation and test
X_validation_imputed_array = imputer.transform(X_validation)
X_test_imputed_array = imputer.transform(X_test)

# Convert the results back into dataframes
X_train_imputed = pd.DataFrame(
    X_train_imputed_array,
    columns=descriptor_columns,
    index=X_train.index
)

X_validation_imputed = pd.DataFrame(
    X_validation_imputed_array,
    columns=descriptor_columns,
    index=X_validation.index
)

X_test_imputed = pd.DataFrame(
    X_test_imputed_array,
    columns=descriptor_columns,
    index=X_test.index
)

print("Missing values after imputation:")

print(
    "Training:",
    X_train_imputed.isna().sum().sum()
)

print(
    "Validation:",
    X_validation_imputed.isna().sum().sum()
)

print(
    "Test:",
    X_test_imputed.isna().sum().sum()
)

In [ ]:
# Remove descriptors that have the same value for every training molecule

variance_filter = VarianceThreshold(threshold=0.0)

# Learn which descriptors are constant using training data only
variance_filter.fit(X_train_imputed)

# Names of descriptors that are not constant
descriptors_after_variance = X_train_imputed.columns[
    variance_filter.get_support()
].tolist()

# Keep the same descriptors in all three datasets
X_train_variance = X_train_imputed[
    descriptors_after_variance
].copy()

X_validation_variance = X_validation_imputed[
    descriptors_after_variance
].copy()

X_test_variance = X_test_imputed[
    descriptors_after_variance
].copy()

constant_descriptors_removed = (
    len(descriptor_columns) -
    len(descriptors_after_variance)
)

print(
    "Descriptors before constant removal:",
    len(descriptor_columns)
)

print(
    "Constant descriptors removed:",
    constant_descriptors_removed
)

print(
    "Descriptors remaining:",
    len(descriptors_after_variance)
)

In [ ]:
# Remove highly correlated descriptors and save the prepared dataset

CORRELATION_CUTOFF = 0.95

# Calculate descriptor correlations using training data only
correlation_matrix = X_train_variance.corr().abs()

# Keep only the upper half because the correlation matrix repeats itself
upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

# Find descriptors having correlation above 0.95
highly_correlated_descriptors = [
    column
    for column in upper_triangle.columns
    if any(
        upper_triangle[column] > CORRELATION_CUTOFF
    )
]

# Remove the same descriptors from all three datasets
X_train_final = X_train_variance.drop(
    columns=highly_correlated_descriptors
)

X_validation_final = X_validation_variance.drop(
    columns=highly_correlated_descriptors
)

X_test_final = X_test_variance.drop(
    columns=highly_correlated_descriptors
)

print(
    "Highly correlated descriptors removed:",
    len(highly_correlated_descriptors)
)

print(
    "Final number of descriptors:",
    X_train_final.shape[1]
)


# Put the three prepared feature datasets back together
prepared_features = pd.concat(
    [
        X_train_final,
        X_validation_final,
        X_test_final
    ]
).sort_index()

# Combine molecule information with the prepared descriptors
prepared_df = pd.concat(
    [
        model_df[information_columns],
        prepared_features
    ],
    axis=1
)

# Save the final prepared dataset
prepared_df.to_csv(
    "preprocessed_model_data.csv",
    index=False
)

# Save the final descriptor names
selected_descriptors = pd.DataFrame({
    "descriptor": X_train_final.columns
})

selected_descriptors.to_csv(
    "selected_descriptor_names.csv",
    index=False
)

print("\nSaved: preprocessed_model_data.csv")
print("Saved: selected_descriptor_names.csv")
print("Final dataset shape:", prepared_df.shape)